**📏 04 — Quãng đường & Loại xe: hai yếu tố quyết định GIÁ**

> ⚠️ **Ba nguyên tắc xuyên suốt**
> 1. **Tách riêng Uber (`dfU`) và Lyft (`dfL`)** — 2 hãng có công thức giá khác nhau.
> 2. Giá thô bị `quãng đường × loại dịch vụ` chi phối → phải kiểm soát khi xét yếu tố khác.
> 3. Uber không có dữ liệu surge → phân tích hệ số nhân chỉ dùng **Lyft**.

Ba notebook trước cho thấy khu vực (14–17%), giờ (~5%), thời tiết (~5%) đều yếu.
Notebook này mổ xẻ **cặp yếu tố chiếm 70–76%**.

Giả thuyết làm việc:
```
giá cuối = giá cơ sở(loại xe, quãng đường) × hệ số nhân
```

**0. Nạp dữ liệu**

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, ".")
import importlib, _common; importlib.reload(_common)
from _common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt

setup()
df, dfU, dfL = load()

BINS = np.linspace(0, df.distance.max(), 16)
XLIM = (0, df.distance.max()*1.02)

**1. 🔍 Nghịch lý mở đầu: nhìn tổng thể thì quãng đường "yếu"**

In [ ]:
r_all = np.corrcoef(df.distance, df.price)[0,1]
print(f"r(quang duong, gia) tren TAT CA du lieu = {r_all:.3f}   <- chi 0.35?!")
print()
for ten, dd in [("Uber", dfU), ("Lyft", dfL)]:
    print(f"  {ten}: r = {np.corrcoef(dd.distance, dd.price)[0,1]:.3f}")
print()
print("Nhung neu tach theo TUNG loai xe:")
for ten, dd in [("Uber", dfU), ("Lyft", dfL)]:
    rs = [np.corrcoef(dd[dd.name==n].distance, dd[dd.name==n].price)[0,1]
          for n in sorted(dd.name.unique())]
    print(f"  {ten}: r = {min(rs):.3f} - {max(rs):.3f}  (TB {np.mean(rs):.3f})")
print()
print("=> Gop 12 hang xe lai lam TRIET TIEU quan he. Day la nghich ly Simpson.")

In [ ]:
s = df.sample(30000, random_state=0)
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
ax[0].scatter(s.distance, s.price, s=5, alpha=.12, color=MUT, edgecolors="none")
xs = np.linspace(0, 8, 50)
ax[0].plot(xs, np.polyval(np.polyfit(df.distance, df.price, 1), xs), color=RED, lw=2.5)
ax[0].set_title(f"Gop TAT CA (r={r_all:.3f}) — dam may hon don", fontweight="bold")
ax[0].set_xlabel("Quang duong (dam)"); ax[0].set_ylabel("Gia (USD)")

svc = df.name.value_counts().index.tolist()
for c, n in zip(plt.cm.tab20(np.linspace(0, 1, len(svc))), svc):
    d = df[df.name == n]
    ds = d.sample(min(2000, len(d)), random_state=0)
    ax[1].scatter(ds.distance, ds.price, s=4, alpha=.15, color=c, edgecolors="none")
    ax[1].plot(xs, np.polyval(np.polyfit(d.distance, d.price, 1), xs), color=c, lw=2, label=n)
ax[1].set_ylim(0, 70); ax[1].set_xlabel("Quang duong (dam)")
ax[1].set_title("Tach theo LOAI XE — hien ra 12 duong thang", fontweight="bold")
ax[1].legend(fontsize=7, ncol=2, frameon=False)
for a in ax: a.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

**2. ⭐ Giải mã công thức giá của từng hạng xe**

Khớp hồi quy tuyến tính `giá = a + b × quãng đường` cho mỗi loại xe.

In [ ]:
rows = []
for ten, dd in [("Uber", dfU), ("Lyft", dfL)]:
    for n in sorted(dd.name.unique()):
        d = dd[dd.name == n]
        b_, a_ = np.polyfit(d.distance, d.price, 1)
        pred = a_ + b_*d.distance
        r2 = 1 - ((d.price-pred)**2).sum()/((d.price-d.price.mean())**2).sum()
        rows.append({"Hãng": ten, "Dịch vụ": n,
                     "Phí mở cửa (USD)": round(a_, 2),
                     "Đơn giá (USD/dặm)": round(b_, 2),
                     "R²": round(r2, 3), "Số chuyến": len(d)})
form = pd.DataFrame(rows).sort_values(["Hãng","Đơn giá (USD/dặm)"], ascending=[True, False])
display(form.reset_index(drop=True))
print("Cong thuc: gia = [Phi mo cua] + [Don gia] x [Quang duong]")
print(f"Don gia chenh nhau {form['Đơn giá (USD/dặm)'].max()/form['Đơn giá (USD/dặm)'].min():.1f} lan "
      f"({form['Đơn giá (USD/dặm)'].min():.2f} -> {form['Đơn giá (USD/dặm)'].max():.2f} USD/dam)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for i, (col, t, c) in enumerate([("Đơn giá (USD/dặm)","Don gia moi dam",BLUE),
                                  ("Phí mở cửa (USD)","Phi mo cua",GREEN)]):
    f = form.sort_values(col)
    cols = [MAU_HANG[h] for h in f["Hãng"]]
    ax[i].barh(range(len(f)), f[col], color=cols, zorder=3)
    ax[i].set_yticks(range(len(f)))
    ax[i].set_yticklabels([f'{r["Dịch vụ"]} ({r["Hãng"][0]})' for _, r in f.iterrows()], fontsize=8.5)
    for j, v in enumerate(f[col]): ax[i].text(v*1.01, j, f"{v:.2f}", va="center", fontsize=7.5)
    ax[i].set_title(t, fontweight="bold"); ax[i].grid(axis="y", visible=False)
import matplotlib.patches as mp
ax[0].legend(handles=[mp.Patch(color=MAU_HANG[h], label=h) for h in ["Uber","Lyft"]],
             fontsize=8, frameon=False, loc="lower right")
fig.tight_layout(); plt.show()

**3. 📊 Bóc tách phương sai — hoàn tất bảng xếp hạng**

In [ ]:
print("Pho gia thu hep the nao khi lan luot co dinh tung yeu to?")
print()
kq = {}
for ten, dd in [("Uber", dfU), ("Lyft", dfL)]:
    b = pd.cut(dd.distance, BINS)
    tho = dd.price.std()
    qd  = dd.groupby(b, observed=True).price.std().mean()
    xe  = dd.groupby([b, "name"], observed=True).price.std().mean()
    print(f"  {ten}")
    print(f"     khong co dinh gi                 : {tho:>5.2f} USD")
    print(f"     + co dinh QUANG DUONG            : {qd:>5.2f} USD   (giam {(1-qd/tho)*100:.0f}%)")
    print(f"     + co dinh LOAI XE                : {xe:>5.2f} USD   (giam them {(1-xe/qd)*100:.0f}%)")
    print(f"     -> con lai chua giai thich duoc  : {xe/tho*100:.0f}% pho gia ban dau")
    print()
    kq[ten] = (tho, qd, xe)

print("BANG XEP HANG CUOI CUNG (% pho gia tho giai thich duoc):")
print("     LOAI XE + QUANG DUONG   70-76%")
print("     KHU VUC                 14-17%")
print("     GIO                      ~5%")
print("     THOI TIET                ~5%")

**4. 🧪 Kiểm chứng bằng model: cần bao nhiêu trường?**

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

bo_feature = {
    "chi distance"            : ["distance"],
    "chi name"                : ["name"],
    "distance + name"         : ["distance","name"],
    "+ source, destination"   : ["distance","name","source","destination"],
    "+ gio"                   : ["distance","name","source","destination","hour_local"],
    "+ thoi tiet"             : ["distance","name","source","destination","hour_local",
                                 "short_summary","temperature","precipIntensity"],
}
res = []
for ten, dd in [("Uber", dfU), ("Lyft", dfL)]:
    d = dd.sample(min(150000, len(dd)), random_state=0)
    for lab, cols in bo_feature.items():
        X = d[cols].copy(); cats = []
        for c in cols:
            if not pd.api.types.is_numeric_dtype(X[c]):
                X[c] = X[c].astype("category"); cats.append(c)
        Xtr, Xte, ytr, yte = train_test_split(X, d.price, test_size=.25, random_state=42)
        m = HistGradientBoostingRegressor(max_iter=200, learning_rate=.06,
                categorical_features=cats if cats else None, random_state=42).fit(Xtr, ytr)
        p = m.predict(Xte)
        res.append({"Hãng": ten, "Bộ feature": lab, "Số trường": len(cols),
                    "R²": round(r2_score(yte, p), 4),
                    "MAE": round(mean_absolute_error(yte, p), 3)})
rr = pd.DataFrame(res)
display(rr.pivot(index="Bộ feature", columns="Hãng", values="R²")
          .reindex(list(bo_feature)))

fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(bo_feature))
for i, ten in enumerate(["Uber","Lyft"]):
    v = rr[rr["Hãng"]==ten].set_index("Bộ feature")["R²"].reindex(list(bo_feature))
    ax.barh(y + (i-.5)*.38, v, height=.38, color=MAU_HANG[ten], label=ten, zorder=3)
    for j, val in enumerate(v): ax.text(val+.008, y[j]+(i-.5)*.38, f"{val:.3f}",
                                        va="center", fontsize=7.5)
ax.set_yticks(y); ax.set_yticklabels(list(bo_feature), fontsize=9)
ax.set_xlim(0, 1.06); ax.set_xlabel("R²"); ax.legend(frameon=False)
ax.set_title("Chi 2 truong da gan nhu tai tao duoc gia", fontweight="bold")
ax.grid(axis="y", visible=False); fig.tight_layout(); plt.show()

for ten in ["Uber","Lyft"]:
    v = rr[rr["Hãng"]==ten].set_index("Bộ feature")["R²"]
    print(f"{ten}: distance+name = {v['distance + name']:.4f} | "
          f"them 5 truong nua = {v['+ thoi tiet']:.4f} "
          f"(+{v['+ thoi tiet']-v['distance + name']:.4f})")

**5. ⭐ Tách giá cơ sở khỏi hệ số nhân**

Kiểm chứng công thức `giá = giá cơ sở × hệ số nhân` bằng cách **suy ngược**:
học giá cơ sở từ các chuyến **không surge**, rồi tính `hệ số ngầm = giá ÷ giá cơ sở`.

In [ ]:
_, du_doan_co_so = base_price_model(df)
df["base_pred"] = du_doan_co_so(df)
df["mult_ngam"] = df.price / df.base_pred
L2 = df[df.cab_type == "Lyft"]

g = L2.groupby("surge_multiplier").mult_ngam.agg(["size","mean","median","std"])
print("He so nhan SUY NGUOC vs he so THAT (Lyft):")
display(g.round(3))
print(f"r(surge that, he so ngam) = {np.corrcoef(L2.surge_multiplier, L2.mult_ngam)[0,1]:.4f}")
print("=> Cong thuc gia = gia_co_so x he_so_nhan duoc XAC NHAN tren du lieu that.")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].plot(g.index, g["mean"], "o-", color=RED, lw=2, ms=6, label="he so ngam (do duoc)")
ax[0].plot(g.index, g.index, "--", color=MUT, lw=1.5, label="duong ly tuong y=x")
ax[0].set_xlabel("surge_multiplier THAT"); ax[0].set_ylabel("he so ngam TB")
ax[0].set_title("Suy nguoc he so nhan co chinh xac khong?", fontweight="bold")
ax[0].legend(fontsize=8, frameon=False)

ns = df[df.surge_multiplier == 1.0]
for ten in ["Uber","Lyft"]:
    s_ = ns[ns.cab_type == ten].mult_ngam
    ax[1].hist(s_, bins=60, range=(.6, 1.8), alpha=.55,
               color=MAU_HANG[ten], label=f"{ten} (std={s_.std():.3f})")
ax[1].axvline(1.0, color="#222", lw=1.5, ls="--")
ax[1].set_xlabel("he so ngam"); ax[1].set_ylabel("So chuyen")
ax[1].set_title("Chuyen KHONG surge: Uber vs Lyft", fontweight="bold")
ax[1].legend(fontsize=8, frameon=False)
for a in ax: a.grid(axis="x", visible=False)
fig.tight_layout(); plt.show()
print()
print("Hai phan bo TRUNG KHIT -> Uber KHONG co surge an de khai thac.")

**6. 🔬 Sàn nhiễu: vì sao model không thể chính xác hơn?**

Cùng loại xe, cùng quãng đường **chính xác tới 0,01 dặm** — giá vẫn lệch. Truy nguồn.

In [ ]:
print("Do lech gia con lai khi siet dan dieu kien:")
print()
for ten, dd in [("Uber", dfU), ("Lyft", dfL)]:
    d2 = dd.assign(d2=dd.distance.round(2))
    b = pd.cut(dd.distance, BINS)
    a1 = dd.groupby(b, observed=True).price.std().mean()
    a2 = dd.groupby([b, "name"], observed=True).price.std().mean()
    g3 = d2.groupby(["d2","name"], observed=True).price.agg(["std","size"])
    g3 = g3[g3["size"] >= 5]
    a3 = g3["std"].mean()
    g4 = d2.groupby(["d2","name","source","destination"], observed=True).price.agg(["std","size"])
    g4 = g4[g4["size"] >= 3]
    print(f"  {ten}")
    print(f"     dai quang duong 0.5 dam              : {a1:>5.2f} USD")
    print(f"     + loai xe                            : {a2:>5.2f} USD")
    print(f"     + quang duong CHINH XAC (0.01 dam)   : {a3:>5.2f} USD")
    print(f"     + tuyen duong                        : {g4['std'].mean():>5.2f} USD")
    print(f"     -> ty le nhom co gia GIONG HET       : {(g3['std']==0).mean()*100:>5.1f}%")
    print()
print("=> Van con ~2-3 USD do lech KHONG giai thich duoc bang bat ky truong nao.")
print("   Day la SAN NHIEU cua bo du lieu -> gioi han tren cua moi model.")

In [ ]:
# Kiem chung: san nhieu nay co phai surge an khong?
U2 = dfU.assign(d2=dfU.distance.round(2))
gmin = U2.groupby(["d2","name"], observed=True).price.transform("min")
U2 = U2.assign(ty_le=U2.price/gmin)
h = U2.groupby("hour_local").ty_le.mean()
print("Uber — ty le gia so voi muc RE NHAT cung (quang duong, loai xe), theo gio:")
print("  " + " ".join(f"{v:.2f}" for v in h.values))
print(f"  Thap nhat {h.min():.3f} (gio {h.idxmin()}) | Cao nhat {h.max():.3f} (gio {h.idxmax()})"
      f" | bien do {h.max()-h.min():.3f}")
print()
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(h.index, h.values, "o-", color=MAU_HANG["Uber"], lw=2, ms=5)
ax.set_ylim(h.mean()-.05, h.mean()+.05)
ax.set_xlabel("Gio"); ax.set_ylabel("Ty le gia TB"); ax.set_xticks(range(24))
ax.set_title("Neu la surge an thi duong nay phai NHO LEN o gio cao diem", fontweight="bold")
ax.grid(axis="x", visible=False); fig.tight_layout(); plt.show()
print("Duong PHANG TUYET DOI qua 24 gio -> KHONG phai surge an, KHONG phai tac duong.")
print("Nhieu kha nang la thuoc tinh chuyen di ma bo du lieu khong ghi lai")
print("(diem don/tra chinh xac, tuyen thuc te, thoi gian cho, khuyen mai...).")

**7. 📌 Kết luận**

In [ ]:
print("="*70); print("QUANG DUONG & LOAI XE — KET LUAN"); print("="*70)
print(f"1) NGHICH LY SIMPSON")
print(f"   - Gop tat ca: r(quang duong, gia) = {r_all:.3f}  (trong nhu yeu)")
print(f"   - Tach theo loai xe: r = 0.7-0.9  -> quan he rat manh")
print()
print(f"2) CONG THUC GIA da giai ma duoc cho ca 12 dich vu")
print(f"   gia = [phi mo cua] + [don gia] x [quang duong]")
print(f"   Don gia chenh {form['Đơn giá (USD/dặm)'].max()/form['Đơn giá (USD/dặm)'].min():.1f} lan giua hang re nhat va dat nhat")
print()
print(f"3) CHI 2 TRUONG la du")
for ten in ["Uber","Lyft"]:
    v = rr[rr["Hãng"]==ten].set_index("Bộ feature")["R²"]
    print(f"   {ten}: distance+name -> R2={v['distance + name']:.3f}; "
          f"them 5 truong nua chi +{v['+ thoi tiet']-v['distance + name']:.3f}")
print()
print(f"4) SAN NHIEU ~2-3 USD")
print(f"   - Cung loai xe, cung quang duong chinh xac -> gia van lech")
print(f"   - Ty le gia PHANG theo gio -> khong phai surge an / tac duong")
print(f"   - Day la gioi han tren: moi model se dung o R2 ~0.93")
print()
print("5) KHUYEN NGHI BO FEATURE CHO MODEL GIA:")
print("   BAT BUOC : name, distance")
print("   TUY CHON : source, destination  (+0.005 R2)")
print("   BO       : gio, thoi tiet       (+0.000)")
print("   LUU Y    : train RIENG cho tung hang (cong thuc gia khac nhau)")